# Qwen3-TTS Standalone Sermon Generator

This notebook implements the Qwen3-TTS model for high-quality, expressive voice cloning. 
It is configured to run on Kaggle GPUs (T4 or P100) and includes an automated "AI Listener" for quality verification using Whisper.

In [ ]:
# 1. Setup Environment
import os
import subprocess
import sys

def run_cmd(cmd):
    print(f"> {cmd}")
    subprocess.run(cmd, shell=True, check=True)

print("Updating system and installing sox...")
run_cmd("apt-get update && apt-get install -y sox libsox-fmt-all")

print("Installing specific Python dependencies...")
# Note: transformers 4.57.3 is the ONLY specific requirement from the Qwen repo.
# We avoid force-reinstalling torch to prevent breaking torchvision/papermill.
run_cmd("pip install transformers==4.57.3 onnxruntime einops sox openai-whisper")

print("Environment Setup Complete.")

In [ ]:
# 3. Clone Implementation and Download Models
import os
from pathlib import Path
from huggingface_hub import snapshot_download

print("Cloning implementation...")
!git clone https://github.com/flybirdxx/ComfyUI-Qwen-TTS temp_repo
!cp -r temp_repo/qwen_tts . 
!rm -rf temp_repo

print("Downloading models (1.7B Base and Tokenizer)...")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

snapshot_download(repo_id="Qwen/Qwen3-TTS-Tokenizer-12Hz", local_dir=MODELS_DIR / "Qwen3-TTS-Tokenizer-12Hz")
snapshot_download(repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base", local_dir=MODELS_DIR / "Qwen3-TTS-12Hz-1.7B-Base")

print("Setup Complete.")

In [ ]:
# 4. Generation Script
import torch
import soundfile as sf
import whisper
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

def generate_and_verify(text, ref_audio_path, ref_text, output_path="output.wav"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32
    
    print(f"Loading Qwen3-TTS Model on {device}...")
    model_path = "models/Qwen3-TTS-12Hz-1.7B-Base"
    model = Qwen3TTSModel.from_pretrained(model_path, device_map=device, torch_dtype=dtype)
    
    print("Generating Audio...")
    wavs, sr = model.generate_voice_clone(
        text=text,
        ref_audio=ref_audio_path,
        ref_text=ref_text,
        language="english",
        temperature=0.8, # Expressive
        top_p=0.95,
        repetition_penalty=1.1
    )
    
    sf.write(output_path, wavs[0], sr)
    print(f"✅ Audio saved to {output_path}")
    
    # --- AI Listener Verification ---
    print("\n--- AI Listener Verification ---")
    v_model = whisper.load_model("base")
    result = v_model.transcribe(output_path)
    transcription = result["text"].strip()
    
    print(f"TRANSCRIBED: {transcription[:200]}...")
    
    return output_path

# --- Execution ---
SERMON_TEXT = """
[solemn]
Yet, as if to show that Jesus the Savior is also Jesus the Judge, one gleam of justice must dart forth. Where shall mercy direct its fall? See, my brethren, it glances not upon a man, but lights upon an unconscious, unsuffering thing--a tree. The curse, if we may call it a curse at all, did not fall on man or beast, or even the smallest insect; its bolt falls harmlessly upon a fig tree by the wayside. It bore upon itself the signs of barrenness, and perhaps was no one's property; little, therefore, was the loss which any man sustained by the withering of that verdant mockery, while instruction more precious than a thousand acres of fig trees has been left for the benefit of all ages.
"""

# Provide path to your seed audio here (e.g. from a Kaggle Dataset)
# For now, we assume a placeholder path.
REF_AUDIO = "/kaggle/input/spurgeon-seed-audio/spurgeon_seed.wav"
REF_TEXT = "I have gone right to the edge of sin. Some strong temptation has taken hold of both my arms."

if os.path.exists(REF_AUDIO):
    generate_and_verify(SERMON_TEXT, REF_AUDIO, REF_TEXT)
else:
    print(f"❌ Reference audio not found at {REF_AUDIO}. Please attach the dataset to the notebook.")